# 🚀 AL-PRORES Cloud Render Engine (Google Colab GPU Edition)
### Render Animasi Apple ProRes 4444 (Alpha Transparent) & MP4 Preview di Cloud GPU Gratis!

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bayukuncoro-id/al-prores-cloud/blob/main/AL_PRORES_Cloud_Render_Engine.ipynb)

Notebook ini berfungsi sebagai **server cloud render** yang terhubung langsung dengan addon **AL-PRORES Cloud** di Blender Anda.

---
### 📋 CARA PAKAI CEPAT (1-KLIK):
1. **Cukup klik menu `Runtime` > `Run all` (atau tekan `Ctrl + F9`).**
2. Saat diminta izin Google Drive di STEP 2, klik **Connect to Google Drive** (agar video 4K otomatis tersimpan ke Google Drive Anda dalam 3 detik).
3. Di STEP 4, klik tombol **[📋 SALIN URL]**, lalu di Blender Addon klik **[🔗 Sambungkan dari Clipboard]** > Selesai!
---

## ⚡ STEP 1: Cek Spesifikasi Cloud GPU (NVIDIA T4)

In [ ]:
# Periksa apakah GPU NVIDIA aktif
!nvidia-smi
import torch
if torch.cuda.is_available():
    print(f"\n✅ GPU Aktif: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM Tersedia: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("\n⚠️ GPU belum aktif! Silakan buka menu Runtime > Change runtime type > Pilih T4 GPU.")

## 💾 STEP 2: Sambungkan Google Drive (Hasil Render Kilat 3 Detik & Anti Disk Penuh)
*Dengan menyambungkan Google Drive, hasil render video 4K langsung tersimpan ke Google Drive Anda secara instan tanpa perlu menunggu download lambat.*

In [ ]:
from google.colab import drive
import os

USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AL-PRORES Render"

try:
    print("⏳ Menghubungkan Google Drive...")
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    USE_GOOGLE_DRIVE = True
    print("\n✅ GOOGLE DRIVE BERHASIL TERHUBUNG!")
    print(f"📁 Folder Output: My Drive / AL-PRORES Render")
except Exception as e:
    print(f"\n⚠️ Google Drive tidak terhubung ({e}).")
    print("ℹ️ Sistem akan otomatis menggunakan direct download ke Blender sebagai cadangan.")

## 📦 STEP 3: Setup Otomatis (Blender 5.2.0 LTS + FFmpeg + Virtual Framebuffer + Cloudflare)
*Mempersiapkan seluruh engine render grafis headless di cloud.*

In [ ]:
import os, sys, shutil, subprocess, urllib.request, time

print("⏳ [1/4] Mempersiapkan folder kerja...")
os.makedirs("/content/al_prores_server", exist_ok=True)
os.makedirs("/content/al_prores_jobs", exist_ok=True)
os.makedirs("/content/al_prores_results", exist_ok=True)

print("⏳ [2/4] Install FFmpeg + NVIDIA EGL & Virtual Display Driver...")
!apt-get update -qq && apt-get install -y -qq ffmpeg xvfb libgl1-mesa-glx mesa-utils libxi6 libxrender1 libxxf86vm1 libxfixes3 libegl1-mesa-dev
print("✅ FFmpeg dan dependensi grafis headless siap!")

print("⏳ [3/4] Mengunduh Blender 5.2.0 LTS Portable (Linux x64)...")
blender_tar = "/content/blender.tar.xz"
blender_dir = "/content/blender_app"
if not os.path.exists("/content/blender_app/blender"):
    blender_url = "https://download.blender.org/release/Blender5.2/blender-5.2.0-linux-x64.tar.xz"
    print(f"   Mengunduh dari {blender_url}...")
    try:
        subprocess.run(["wget", "-q", "--show-progress", "-O", blender_tar, blender_url], check=True)
    except Exception:
        print("   Fallback ke mirror...")
        blender_url_fb = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
        subprocess.run(["wget", "-q", "--show-progress", "-O", blender_tar, blender_url_fb], check=True)

    print("   Mengekstrak Blender...")
    os.makedirs(blender_dir, exist_ok=True)
    subprocess.run(["tar", "-xf", blender_tar, "-C", blender_dir, "--strip-components=1"], check=True)
    if os.path.exists(blender_tar):
        os.remove(blender_tar)
    print("✅ Blender portable 5.2.0 LTS berhasil diinstall!")
else:
    print("✅ Blender portable 5.2.0 LTS sudah terinstall.")

print("⏳ [4/4] Mengunduh Cloudflare Tunnel (Tanpa akun / 100% Gratis)...")
cf_bin = "/content/cloudflared"
if not os.path.exists(cf_bin):
    cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    subprocess.run(["wget", "-q", "-O", cf_bin, cf_url], check=True)
    subprocess.run(["chmod", "+x", cf_bin], check=True)
    print("✅ Cloudflare Tunnel siap digunakan!")
else:
    print("✅ Cloudflare Tunnel sudah tersedia.")

print("\n🎉 SELURUH SISTEM SIAP! Menjalankan worker di bawah...")

## 🚀 STEP 4: Jalankan Cloud Worker & Jembatan Tunnel
*Cell ini menjalankan server worker dan membuka URL jembatan ke Blender.*

In [ ]:
import os, sys, json, time, shutil, zipfile, threading, subprocess, re, traceback
from http.server import HTTPServer, BaseHTTPRequestHandler
import torch
from IPython.display import HTML, display

# PAKSA BLENDER EEVEE MENGGUNAKAN GPU NVIDIA (Bukan CPU llvmpipe!)
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"
os.environ["EGL_PLATFORM"] = "device"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU / Unknown"
BLENDER_BIN = "/content/blender_app/blender"
JOBS_DIR = "/content/al_prores_jobs"
RESULTS_DIR = "/content/al_prores_results"

# State Penyimpanan Job
jobs = {}
current_render_process = None

def execute_render_job(job_id):
    global current_render_process
    job_dir = os.path.join(JOBS_DIR, job_id)
    manifest_path = os.path.join(job_dir, "manifest.json")
    blend_path = os.path.join(job_dir, "scene.blend")
    
    if not os.path.exists(manifest_path) or not os.path.exists(blend_path):
        jobs[job_id]["status"] = "error"
        jobs[job_id]["error"] = "Missing scene.blend or manifest.json in package"
        return

    try:
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)
    except Exception as e:
        jobs[job_id]["status"] = "error"
        jobs[job_id]["error"] = f"Corrupt manifest: {e}"
        return

    collections = manifest.get("collections", [])
    if not collections:
        jobs[job_id]["status"] = "error"
        jobs[job_id]["error"] = "No collections specified in render manifest"
        return

    jobs[job_id]["status"] = "rendering"
    jobs[job_id]["total_collections"] = len(collections)
    jobs[job_id]["progress"] = 5

    output_png_dir = os.path.join(job_dir, "png_frames")
    os.makedirs(output_png_dir, exist_ok=True)

    isolation_helper_code = """
def is_setup_collection(col):
    name_upper = col.name.upper().strip()
    renderable_types = {'MESH', 'CURVE', 'SURFACE', 'META', 'FONT', 'VOLUME', 'GPENCIL', 'GREASEPENCIL'}
    has_renderable = any(o.type in renderable_types for o in col.all_objects)
    if not has_renderable:
        return True
    setup_names = ['STUDIO', 'LIGHTS', 'LIGHTING', 'CAMERAS', 'ENVIRONMENT', 'ENV', 'RIG', 'SETUP']
    has_lighting_or_cam = any(o.type in {'LIGHT', 'CAMERA'} for o in col.all_objects)
    if name_upper in setup_names and has_lighting_or_cam:
        return True
    return False

def find_layer_collection(layer_col, col_name):
    if not layer_col:
        return None
    if layer_col.collection.name == col_name:
        return layer_col
    for child in layer_col.children:
        found = find_layer_collection(child, col_name)
        if found:
            return found
    return None
"""

    rendered_files = []
    total_coll = len(collections)

    fps = manifest.get("fps", 30)
    q_scale = manifest.get("q_scale", "9")
    w = manifest.get("resolution_x", 1920)
    h = manifest.get("resolution_y", 1080)
    gen_mp4 = manifest.get("generate_mp4_preview", True)
    bg_rgb = manifest.get("preview_bg_color", [1.0, 1.0, 1.0])
    bg_hex = '#{:02x}{:02x}{:02x}'.format(int(bg_rgb[0]*255), int(bg_rgb[1]*255), int(bg_rgb[2]*255))

    for idx, coll_name in enumerate(collections):
        jobs[job_id]["current_collection"] = coll_name
        jobs[job_id]["current_index"] = idx + 1
        base_pct = 5 + int((idx / total_coll) * 85)
        jobs[job_id]["progress"] = base_pct
        jobs[job_id]["message"] = f"Memulai render {coll_name}..."
        print(f"\n🚀 [Job #{job_id}] Rendering Collection ({idx+1}/{total_coll}): {coll_name}")

        shutil.rmtree(output_png_dir, ignore_errors=True)
        os.makedirs(output_png_dir, exist_ok=True)

        coll_script = os.path.join(job_dir, f"render_coll_{idx}.py")
        with open(coll_script, "w", encoding="utf-8") as csf:
            csf.write(f"""
import bpy, json, sys, os

with open(r'{manifest_path}', 'r', encoding='utf-8') as f:
    mf = json.load(f)

scene = bpy.context.scene

# Setup GPU Devices
try:
    prefs = bpy.context.preferences
    cprefs = prefs.addons.get('cycles')
    if cprefs:
        cprefs.preferences.compute_device_type = 'CUDA'
        cprefs.preferences.get_devices()
        for d in cprefs.preferences.devices:
            d.use = True
except Exception as e:
    print(f'Cycles CUDA error: {{e}}')

# Set output settings
scene.render.image_settings.file_format = 'PNG'
scene.render.image_settings.color_mode = 'RGBA'
scene.render.image_settings.color_depth = '8'
scene.render.film_transparent = True
scene.render.resolution_x = mf.get('resolution_x', 1920)
scene.render.resolution_y = mf.get('resolution_y', 1080)
scene.render.resolution_percentage = 100
scene.render.fps = mf.get('fps', 30)
scene.frame_start = mf.get('frame_start', 1)
scene.frame_end = mf.get('frame_end', 250)
scene.render.filepath = r'{output_png_dir}/frame_'

# Isolate collection logic
{isolation_helper_code}

target_col_name = {json.dumps(coll_name)}
target_col = bpy.data.collections.get(target_col_name)
if not target_col:
    print(f'Error: Target collection {{target_col_name}} not found!')
    sys.exit(1)

master_col = scene.collection
view_layer = bpy.context.view_layer

for c in master_col.children:
    lc = find_layer_collection(view_layer.layer_collection, c.name)
    if lc:
        if c.name == target_col_name:
            lc.exclude = False
            c.hide_render = False
        elif is_setup_collection(c):
            lc.exclude = False
            c.hide_render = False
            print(f'Preserved Studio/Rig Collection: {{c.name}}')
        else:
            lc.exclude = True
            c.hide_render = True

target_layer_col = find_layer_collection(view_layer.layer_collection, target_col_name)
if target_layer_col:
    target_layer_col.exclude = False
target_col.hide_render = False

# Ensure active camera
if not scene.camera:
    for obj in bpy.data.objects:
        if obj.type == 'CAMERA':
            scene.camera = obj
            print(f'Assigned fallback camera: {{obj.name}}')
            break

print(f'=== Starting render: {{target_col_name}} (Frames {{scene.frame_start}}-{{scene.frame_end}}) ===')
bpy.ops.render.render(animation=True)
print('=== Finished render collection ===')
""")

        # Eksekusi Render dengan Virtual Framebuffer (xvfb-run) & Akselerasi Hardware GPU
        cmd = [
            "xvfb-run", "-a", "-s", "-screen 0 1920x1080x24",
            BLENDER_BIN, "-b", blend_path,
            "--python", coll_script
        ]
        env = os.environ.copy()
        current_render_process = subprocess.Popen(
            cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )

        total_frames = max(1, manifest.get("frame_end", 250) - manifest.get("frame_start", 1) + 1)
        rendered_in_coll = 0

        for line in current_render_process.stdout:
            if "Append frame" in line or "Fra:" in line or "Rendered" in line:
                rendered_in_coll += 1
                frame_pct = min(1.0, rendered_in_coll / total_frames)
                coll_pct = base_pct + int(frame_pct * (85 / total_coll))
                jobs[job_id]["progress"] = min(92, coll_pct)
                jobs[job_id]["message"] = f"[{coll_name}] Frame {rendered_in_coll}/{total_frames}"

        current_render_process.wait()
        ret = current_render_process.returncode
        current_render_process = None

        if ret != 0:
            jobs[job_id]["status"] = "error"
            jobs[job_id]["error"] = f"Blender render failed with code {ret} on {coll_name}"
            return

        # Encode hasil render PNG sequence ke Apple ProRes 4444 (.mov transparan)
        jobs[job_id]["message"] = f"Encoding ProRes 4444 for {coll_name}..."
        clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', coll_name)
        out_mov = os.path.join(job_dir, f"{clean_name}_prores.mov")
        
        ff_cmd = [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-i", os.path.join(output_png_dir, "frame_%04d.png"),
            "-c:v", "prores_ks",
            "-profile:v", "4444",
            "-pix_fmt", "yuva444p10le",
            "-vendor", "apl0",
            "-q:v", str(q_scale),
            "-color_primaries", "bt709",
            "-color_trc", "bt709",
            "-colorspace", "bt709",
            "-metadata:s:v:0", "alpha_mode=premultiplied",
            out_mov
        ]
        ff_res = subprocess.run(ff_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        if ff_res.returncode != 0:
            jobs[job_id]["status"] = "error"
            jobs[job_id]["error"] = f"FFmpeg ProRes error: {ff_res.stdout[-300:]}"
            return
        rendered_files.append(out_mov)

        # Generate MP4 Preview jika diminta
        if gen_mp4:
            jobs[job_id]["message"] = f"Encoding MP4 Preview for {coll_name}..."
            out_mp4 = os.path.join(job_dir, f"{clean_name}_preview.mp4")
            bg_type = manifest.get("preview_bg_type", "COLOR")
            bg_img_arc = manifest.get("preview_bg_image", "")
            bg_img_file = os.path.join(job_dir, bg_img_arc) if bg_img_arc else ""

            if bg_type == "IMAGE" and bg_img_file and os.path.exists(bg_img_file):
                mp4_cmd = [
                    "ffmpeg", "-y",
                    "-loop", "1",
                    "-i", bg_img_file,
                    "-i", out_mov,
                    "-filter_complex", f"[0:v]scale={w}:{h}:force_original_aspect_ratio=increase,crop={w}:{h}[bg];[bg][1:v]overlay=shortest=1,format=yuv420p",
                    "-c:v", "libx264",
                    "-preset", "fast",
                    "-crf", "20",
                    "-movflags", "+faststart",
                    "-shortest",
                    out_mp4
                ]
            else:
                mp4_cmd = [
                    "ffmpeg", "-y",
                    "-i", out_mov,
                    "-filter_complex", f"color=c={bg_hex}:s={w}x{h}[bg];[bg][0:v]overlay=shortest=1,format=yuv420p",
                    "-c:v", "libx264",
                    "-preset", "fast",
                    "-crf", "20",
                    "-movflags", "+faststart",
                    out_mp4
                ]
            subprocess.run(mp4_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            if os.path.exists(out_mp4):
                rendered_files.append(out_mp4)

        # HEMAT DISK: Hapus file PNG frame sequence setelah MOV & MP4 selesai di-encode
        shutil.rmtree(output_png_dir, ignore_errors=True)
        os.makedirs(output_png_dir, exist_ok=True)

    # SIMPAN KE GOOGLE DRIVE (JIKA TERHUBUNG)
    drive_saved = False
    drive_folder_name = ""
    if os.path.exists("/content/drive/MyDrive"):
        try:
            jobs[job_id]["message"] = "Menyimpan ke Google Drive..."
            clean_proj_name = re.sub(r'[^a-zA-Z0-9_-]', '_', manifest.get("project_name", "Render"))
            job_folder_name = f"{clean_proj_name}_{job_id}"
            drive_dest_dir = os.path.join("/content/drive/MyDrive/AL-PRORES Render", job_folder_name)
            os.makedirs(drive_dest_dir, exist_ok=True)
            for rf in rendered_files:
                dest_file = os.path.join(drive_dest_dir, os.path.basename(rf))
                shutil.copy2(rf, dest_file)
            drive_saved = True
            drive_folder_name = f"AL-PRORES Render/{job_folder_name}"
            print(f"\n💾 [Google Drive] Berhasil menyimpan {len(rendered_files)} video ke '{drive_folder_name}'!")
        except Exception as e:
            print(f"\n⚠️ Gagal menyalin ke Google Drive ({e}), fallback ke ZIP download.")
            drive_saved = False

    # Packing hasil render ke ZIP result sebagai backup / fallback
    jobs[job_id]["message"] = "Membuat paket hasil render (ZIP)..."
    result_zip = os.path.join(RESULTS_DIR, f"result_{job_id}.zip")
    with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for rf in rendered_files:
            zf.write(rf, arcname=os.path.basename(rf))

    # PEMBERSIHAN FINAL DISK COLAB (Hapus frame PNG & script temp)
    try:
        shutil.rmtree(output_png_dir, ignore_errors=True)
        for f in os.listdir(job_dir):
            if f.endswith(".py") or f.endswith(".png"):
                try: os.remove(os.path.join(job_dir, f))
                except: pass
    except Exception:
        pass

    jobs[job_id]["status"] = "completed"
    jobs[job_id]["progress"] = 100
    msg_done = f"Render Selesai! ({len(rendered_files)} files)"
    if drive_saved:
        msg_done += f" -> Tersimpan di Google Drive ({drive_folder_name})"
    jobs[job_id]["message"] = msg_done
    jobs[job_id]["result_zip"] = result_zip
    jobs[job_id]["drive_saved"] = drive_saved
    jobs[job_id]["drive_folder"] = drive_folder_name
    jobs[job_id]["files"] = [os.path.basename(rf) for rf in rendered_files]
    print(f"\n🎉 [Job #{job_id}] SELESAI! {msg_done}")


class ColabWorkerHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

    def do_GET(self):
        if self.path == "/status":
            drive_status = os.path.exists("/content/drive/MyDrive")
            data = {
                "status": "ok",
                "gpu": GPU_NAME,
                "blender_version": "5.2.0 LTS",
                "google_drive": drive_status,
                "active_jobs": len([j for j in jobs.values() if j.get("status") == "rendering"])
            }
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Access-Control-Allow-Origin", "*")
            self.end_headers()
            self.wfile.write(json.dumps(data).encode("utf-8"))

        elif self.path.startswith("/job_status"):
            from urllib.parse import urlparse, parse_qs
            qs = parse_qs(urlparse(self.path).query)
            job_id = qs.get("job_id", [""])[0]
            if job_id in jobs:
                resp = jobs[job_id]
            else:
                resp = {"status": "not_found", "job_id": job_id}
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Access-Control-Allow-Origin", "*")
            self.end_headers()
            self.wfile.write(json.dumps(resp).encode("utf-8"))

        elif self.path.startswith("/job_download"):
            from urllib.parse import urlparse, parse_qs
            qs = parse_qs(urlparse(self.path).query)
            job_id = qs.get("job_id", [""])[0]
            res_zip = os.path.join(RESULTS_DIR, f"result_{job_id}.zip")
            if os.path.exists(res_zip):
                self.send_response(200)
                self.send_header("Content-Type", "application/zip")
                self.send_header("Content-Disposition", f'attachment; filename="result_{job_id}.zip"')
                self.send_header("Content-Length", str(os.path.getsize(res_zip)))
                self.end_headers()
                with open(res_zip, "rb") as f:
                    shutil.copyfileobj(f, self.wfile)
            else:
                self.send_response(404)
                self.end_headers()
                self.wfile.write(b"Result ZIP not found or render in progress")
        else:
            self.send_response(404)
            self.end_headers()

    def do_POST(self):
        if self.path == "/render_job":
            job_id = self.headers.get("X-Job-ID", str(int(time.time())))
            length = int(self.headers.get("Content-Length", 0))
            zip_bytes = self.rfile.read(length)

            job_dir = os.path.join(JOBS_DIR, job_id)
            os.makedirs(job_dir, exist_ok=True)
            zip_save = os.path.join(job_dir, "upload.zip")
            with open(zip_save, "wb") as f:
                f.write(zip_bytes)

            try:
                with zipfile.ZipFile(zip_save, "r") as zf:
                    zf.extractall(job_dir)
            except Exception as e:
                self.send_response(400)
                self.end_headers()
                self.wfile.write(json.dumps({"error": f"Invalid ZIP archive: {e}"}).encode("utf-8"))
                return

            jobs[job_id] = {
                "job_id": job_id,
                "status": "queued",
                "progress": 0,
                "message": "Antrean diterima di Google Colab GPU",
                "created_at": time.time()
            }

            t = threading.Thread(target=execute_render_job, args=(job_id,), daemon=True)
            t.start()

            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"status": "queued", "job_id": job_id}).encode("utf-8"))

        elif self.path.startswith("/job_cancel"):
            global current_render_process
            if current_render_process:
                try:
                    current_render_process.terminate()
                except Exception:
                    pass
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(b'{"status": "cancelled"}')
        else:
            self.send_response(404)
            self.end_headers()

# Jalankan HTTP Worker Server di port 8000
server = HTTPServer(("0.0.0.0", 8000), ColabWorkerHandler)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()

# Jalankan Cloudflare Tunnel ke port 8000
tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Dapatkan URL Cloudflare publik
public_url = None
for i in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("="*75)
    print("🚀 AL-PRORES CLOUD RENDER WORKER AKTIF & SIAP TERHUBUNG!")
    print(f"⚡ GPU Aktif: {GPU_NAME} (NVIDIA Hardware Acceleration Enabled)")
    drive_ok = os.path.exists("/content/drive/MyDrive")
    if drive_ok:
        print("💾 Google Drive: TERHUBUNG (Hasil otomatis disimpan ke Google Drive dalam 3 detik)")
    else:
        print("💾 Google Drive: Tidak terhubung (Hasil diunduh lewat direct tunnel)")
    print("-"*75)
    print(f"URL Jembatan: {public_url}\n")
    print("Klik tombol [📋 SALIN URL] di bawah ini, lalu di Blender klik [🔗 Sambungkan dari Clipboard]!")
    print("="*75)
    
    html_ui = f"""
    <div style="background: linear-gradient(135deg, #181824, #272738); padding: 22px; border-radius: 12px; border: 2px solid #6366f1; max-width: 650px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 8px 24px rgba(0,0,0,0.4); margin: 15px 0;">
        <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 12px;">
            <h3 style="color: #ffffff; margin: 0; font-size: 18px; display: flex; align-items: center; gap: 8px;">
                🚀 <span>AL-PRORES Cloud Worker Siap!</span>
            </h3>
            <span style="background: {'#10b981' if drive_ok else '#f59e0b'}; color: white; padding: 4px 10px; border-radius: 20px; font-size: 11px; font-weight: bold;">
                {'💾 GOOGLE DRIVE ON' if drive_ok else '⚠️ DRIVE OFF (TUNNEL)'}
            </span>
        </div>
        <p style="color: #cbd5e1; font-size: 13.5px; line-height: 1.5; margin: 0 0 16px 0;">
            Klik tombol di bawah ini untuk menyalin URL secara otomatis, lalu di Blender Addon klik tombol <b>[🔗 Sambungkan dari Clipboard]</b>:
        </p>
        <div style="display: flex; gap: 10px; align-items: center;">
            <input type="text" id="cf_url_box" value="{public_url}" readonly style="flex: 1; padding: 11px 14px; border-radius: 8px; border: 1px solid #475569; background: #0f172a; color: #38bdf8; font-family: monospace; font-size: 14px; outline: none; font-weight: 500;">
            <button onclick="navigator.clipboard.writeText('{public_url}'); var b = document.getElementById('cf_copy_btn'); b.innerText='✅ TERSALIN!'; b.style.background='#10b981'; setTimeout(() => {{ b.innerText='📋 SALIN URL'; b.style.background='#6366f1'; }}, 2500);" id="cf_copy_btn" style="background: #6366f1; color: white; border: none; padding: 11px 22px; border-radius: 8px; font-weight: bold; font-size: 14px; cursor: pointer; transition: all 0.2s;">
                📋 SALIN URL
            </button>
        </div>
    </div>
    """
    display(HTML(html_ui))
else:
    print("⚠️ Gagal mendapatkan URL Cloudflare otomatis. Coba restart cell ini.")

# Jaga cell agar tetap berjalan melayani render request
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Worker dihentikan oleh user.")
    server.shutdown()

## 📦 STEP 5 (ALTERNATIF): Render Paket Offline Tanpa Tunnel
*Jika Anda tidak ingin menggunakan live tunnel, Anda bisa mengklik tombol `Export Colab Package (.zip)` di Blender Addon, lalu upload file ZIP tersebut ke file browser Colab di sebelah kiri, kemudian jalankan cell di bawah ini.*

In [ ]:
import glob, os, zipfile, time

uploaded_zips = glob.glob("/content/AL_PRORES_Colab_Package_*.zip") + glob.glob("/content/*.zip")
package_zips = [z for z in uploaded_zips if "al_prores" in os.path.basename(z).lower() or "package" in os.path.basename(z).lower()]

if not package_zips:
    print("⚠️ Tidak ditemukan paket .zip di /content/!")
    print("Silakan upload file 'AL_PRORES_Colab_Package_xxxx.zip' ke folder /content/ lalu jalankan kembali cell ini.")
else:
    pkg = package_zips[0]
    job_id = f"manual_{int(time.time())}"
    job_dir = f"/content/al_prores_jobs/{job_id}"
    os.makedirs(job_dir, exist_ok=True)
    print(f"📦 Mengekstrak {pkg}...")
    with zipfile.ZipFile(pkg, "r") as zf:
        zf.extractall(job_dir)
    
    jobs[job_id] = {"job_id": job_id, "status": "queued", "progress": 0, "message": "Render manual dimulai"}
    print(f"🚀 Memulai render offline dengan GPU {GPU_NAME}...")
    execute_render_job(job_id)
    
    result_file = jobs[job_id].get("result_zip")
    if result_file and os.path.exists(result_file):
        from google.colab import files
        print(f"\n✅ Selesai! Mengunduh otomatis {os.path.basename(result_file)}...")
        files.download(result_file)